# Multiple fuels: octane primary + hydrogen reheat

Two **very different fuels injected at different positions** and burned:
liquid-like **n-octane** (`C8H18`) in the primary zone, then **hydrogen**
(`H2`) injected into the hot products as a **reheat** stage.

This shows the chemistry plumbing does not depend on the fuel:

* you name whatever species you want from the thermochemical data; the network
  transports one conserved **mixture fraction** per distinct injected composition
  (here air, octane, hydrogen), worked out from the element compositions at build time;
* a source specified by species needs no translation table -- its stream is
  reconstructed exactly by a forward blend, so octane and hydrogen coexist with
  no element bookkeeping;
* injecting H2 into the **burnt** (equilibrium) stream re-equilibrates and
  releases more heat -- a reheat with no extra flame element needed.

> **Mixture-fraction transport.** Each feed stream is its own transported scalar,
> so the unburnt reconstruction is unambiguous no matter how many fuels are
> co-mixed -- even two carbon-bearing fuels injected into one unburnt stream
> (which an elemental basis cannot resolve) is fine here.

![Network topology](multiple_fuels_topology.png)

In [ ]:
import os
import tempfile

import numpy as np
import plotly.graph_objects as go

import nefes
from nefes.thermo import ThermoInp, Thermo, EQ_FROZEN, EQ_KERNEL
from nefes.chem import resolve_composition, enthalpy_mass
from nefes.elements import catalog as cat
from nefes.plotting import use_nefes_theme, COLORWAY

use_nefes_theme()

In [ ]:
HEAVY = "C8H18,n-octane"
lib = ThermoInp().library(
    ["O2", "N2", HEAVY, "H2", "CO2", "H2O", "CO", "OH", "H", "O", "NO"]
)
gas = Thermo(lib)
print("elements:", lib.elements)
print("species :", [s.name for s in lib.species])

## Staged network

`air -> octane injector -> flame -> H2 reheat injector -> outlet`. The octane
burns at the flame (frozen -> equilibrium); the H2 is injected into the burnt
equilibrium stream and re-equilibrates (reheat).

In [ ]:
AIR = {"O2": 0.21, "N2": 0.79}
OCT = {HEAVY: 1.0}
H2 = {"H2": 1.0}


def build_staged(mdot_air=1.0, mdot_oct=0.04, mdot_h2=0.008, Tair=400.0, Tfuel=300.0, p=5.0e5, A=0.06):
    """air -> octane injector -> flame -> H2 reheat injector -> outlet.

    Three feed streams (air, octane, H2) are discovered at build time; ``net.solve()``
    seeds every edge by propagating them through the network -- no hand-built guess.
    """
    Yair, _ = resolve_composition(lib, AIR)
    Yoct, _ = resolve_composition(lib, OCT)
    h_air = enthalpy_mass(lib, Yair, Tair)
    h_oct = enthalpy_mass(lib, Yoct, Tfuel)
    m1 = mdot_air + mdot_oct
    m2 = m1 + mdot_h2
    h_mix = (mdot_air * h_air + mdot_oct * h_oct) / m1  # enthalpy scale for h_ref

    nodes = [
        cat.mass_flow_inlet(mdot_air, Tair, composition=AIR, name="air"),
        cat.mass_source(mdot_oct, Tfuel, composition=OCT, name="octane"),
        cat.equilibrium_flame(name="flame"),
        cat.mass_source(mdot_h2, Tfuel, composition=H2, name="H2-reheat"),
        cat.pressure_outlet(p, Tt_backflow=Tair, composition=AIR, name="out"),
    ]
    edges = [(0, 1, A), (1, 2, A), (2, 3, A), (3, 4, A)]
    edge_models = [EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL]
    return nefes.Network(nefes.equilibrium(lib),
                         nodes=nodes, edges=edges, edge_models=edge_models)


net = build_staged()
sol = net.solve()
# the feed streams are discovered at build time and surface as the mixture-fraction labels
print("feed streams (transported mixture fractions):", list(sol.mixture_fractions(0).keys()))
print("converged:", sol.converged, " iterations:", sol.iterations)

## Edge states and the feed-stream mixture fractions

The transported mixture fractions `xi` pick up octane as it is injected and H2 at the reheat stage; the flame burns octane, the H2 stage reheats.

In [ ]:
names = ["air", "air+octane", "burnt(oct)", "burnt+H2"]
E = len(sol.field("T"))
labels = list(sol.mixture_fractions(0).keys())

print(f"{'edge':<12}{'mdot':>9}{'T [K]':>10}{'p [bar]':>10}{'M':>8}")
for e in range(E):
    st = sol.edge(e)
    print(f"{names[e]:<12}{st['mdot']:9.4f}{st['T']:10.1f}{st['p'] / 1e5:10.3f}{st['M']:8.3f}")
print()

# transported mixture fractions xi per edge -- one per feed stream
print("feed-stream mixture fractions xi by edge:")
print(f"{'':<12}" + "".join(f"{nm:>11}" for nm in labels))
for e in range(E):
    xi = sol.mixture_fractions(e)
    print(f"{names[e]:<12}" + "".join(f"{xi[nm]:11.4f}" for nm in labels))
print(f"\nH2 reheat raises T by {sol.edge(3)['T'] - sol.edge(2)['T']:.0f} K")

## Temperature through the stages

In [ ]:
xa = [0, 1, 2, 3]
stations = ["air", "octane in", "flame", "H₂ reheat"]
T = [sol.edge(e)["T"] for e in range(4)]
fig = go.Figure()
fig.add_scatter(x=xa, y=T, mode="lines+markers+text",
                line=dict(color=COLORWAY[3], width=2), marker=dict(size=10),
                text=[f"{nm}<br>{ti:.0f} K" for nm, ti in zip(names, T)],
                textposition="top center", cliponaxis=False)
fig.update_layout(title="Octane primary + H₂ reheat",
                  yaxis_title="static temperature  [K]", showlegend=False, height=440)
fig.update_xaxes(tickmode="array", tickvals=xa, ticktext=stations)
fig.show()

## Hydrogen-reheat sweep

More H2 into the burnt gas -> more reheat (until the residual oxygen is consumed).

In [ ]:
h2s = np.linspace(0.0, 0.02, 11)
Treheat, Tprimary = [], []
for mh in h2s:
    s = build_staged(mdot_h2=max(mh, 1e-9)).solve()
    Tprimary.append(s.edge(2)["T"])
    Treheat.append(s.edge(3)["T"])

fig = go.Figure()
fig.add_scatter(x=h2s, y=Tprimary, mode="lines+markers", name="after octane flame",
                line=dict(color=COLORWAY[4], dash="dash"))
fig.add_scatter(x=h2s, y=Treheat, mode="lines+markers", name="after H₂ reheat", line_color=COLORWAY[3])
fig.update_layout(title="Hydrogen-reheat sweep",
                  xaxis_title="H₂ reheat ṁ  [kg/s]", yaxis_title="temperature  [K]", height=440)
fig.show()

## Export for Nemo

The staged network `net` and its converged mean flow `sol` are already in scope.
Save either to a UI-readable YAML -- `sol.to_yaml(path)` embeds the mean-flow result
fields, `net.to_yaml(path)` writes the topology only -- then open the file in Nemo.

In [ ]:
_out = os.path.join(tempfile.mkdtemp(), "multiple_fuels.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use net.to_yaml(_out) for topology only
print("saved case:", _out)